In [4]:
import pandas as pd
from sqlalchemy import create_engine

df_clean = pd.read_csv("../data/processed/AB_US_2020_cleaned.csv")

# Create a SQLite database file
engine = create_engine("sqlite:///../data/processed/airbnb.db")

df_clean.to_sql("listings", engine, if_exists="replace", index=False)

print("Data loaded into SQLite database successfully.")

Data loaded into SQLite database successfully.


In [5]:
import sqlite3

conn = sqlite3.connect("../data/processed/airbnb.db")
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM listings")
print(cursor.fetchone())

cursor.execute("SELECT * FROM listings LIMIT 5")
for row in cursor.fetchall():
    print(row)

(223610,)
(38585, 'Charming Victorian home - twin beds + breakfast', 165529, 'Evelyne', 'Not Reported', '28804', 35.65146, -82.62792, 'Private room', 60, 1, 138, '2020-02-16', 1.14, 1, 0, 'Asheville')
(80905, 'French Chic Loft', 427027, 'Celeste', 'Not Reported', '28801', 35.59779, -82.5554, 'Entire home/apt', 470, 1, 114, '2020-07-09', 1.03, 11, 288, 'Asheville')
(108061, 'Walk to stores/parks/downtown. Fenced yard/Pets OK', 320564, 'Lisa', 'Not Reported', '28801', 35.6067, -82.55563, 'Entire home/apt', 75, 30, 89, '2019-11-30', 0.81, 2, 298, 'Asheville')
(155305, "Cottage! BonPaul + Sharky's Hostel", 746673, 'BonPaul', 'Not Reported', '28806', 35.57864, -82.59578, 'Entire home/apt', 90, 1, 267, '2020-09-22', 2.39, 5, 0, 'Asheville')
(160594, 'Historic Grove Park', 769252, 'Elizabeth', 'Not Reported', '28801', 35.61442, -82.54127, 'Private room', 125, 30, 58, '2015-10-19', 0.52, 1, 0, 'Asheville')


In [6]:
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='listings'")
schema = cursor.fetchone()[0]
print(schema)

CREATE TABLE listings (
	id BIGINT, 
	name TEXT, 
	host_id BIGINT, 
	host_name TEXT, 
	neighbourhood_group TEXT, 
	neighbourhood TEXT, 
	latitude FLOAT, 
	longitude FLOAT, 
	room_type TEXT, 
	price BIGINT, 
	minimum_nights BIGINT, 
	number_of_reviews BIGINT, 
	last_review TEXT, 
	reviews_per_month FLOAT, 
	calculated_host_listings_count BIGINT, 
	availability_365 BIGINT, 
	city TEXT
)


In [8]:
import pandas as pd

# See the table's structure
pd.read_sql("PRAGMA table_info(listings);", engine)

,cid,name,type,notnull,dflt_value,pk
0,0,id,BIGINT,0,None,0
1,1,name,TEXT,0,None,0
2,2,host_id,BIGINT,0,None,0
3,3,host_name,TEXT,0,None,0
4,4,neighbourhood_group,TEXT,0,None,0
5,5,neighbourhood,TEXT,0,None,0
6,6,latitude,FLOAT,0,None,0
7,7,longitude,FLOAT,0,None,0
8,8,room_type,TEXT,0,None,0
9,9,price,BIGINT,0,None,0


In [9]:
query1 = """
SELECT name, city, price, room_type
FROM listings
WHERE city = 'New York City'
ORDER BY price DESC
LIMIT 10;
"""

result1 = pd.read_sql(query1, engine)
result1

,name,city,price,room_type
0,Carol,New York City,1700,Private room
1,"Watson Hotel, Standard Double Double",New York City,1667,Hotel room
2,"The Watson, Standard Queen",New York City,1667,Hotel room
3,"The Watson Hotel, Standard King",New York City,1667,Hotel room
4,"The Watson Hotel, Accessible Double Double",New York City,1667,Hotel room
5,"The Watson Hotel, Accessible King",New York City,1667,Hotel room
6,"The Watson, Accessible King with Roll In Shower",New York City,1667,Hotel room
7,brooklyn 14 bedroom gated community,New York City,1661,Entire home/apt
8,Magnificent Lakeview Home on Kissena Park in NYC,New York City,1643,Entire home/apt
9,MANHATTAN SUPERBOWL ACCOMODATION,New York City,1600,Entire home/apt


In [10]:
query2 = """
SELECT room_type, AVG(reviews_per_month) as avg_reviews_per_month
FROM listings
GROUP BY room_type
ORDER BY avg_reviews_per_month DESC;
"""

result2 = pd.read_sql(query2, engine)
result2

,room_type,avg_reviews_per_month
0,Entire home/apt,1.217275
1,Hotel room,1.094576
2,Private room,0.972033
3,Shared room,0.569915


In [11]:
query3 = """
SELECT host_id, COUNT(*) as listing_count
FROM listings
GROUP BY host_id
ORDER BY listing_count DESC
LIMIT 10;
"""

result3 = pd.read_sql(query3, engine)
result3

,host_id,listing_count
0,48005494,1215
1,107434423,1172
2,359036978,593
3,8534462,582
4,30283594,332
5,359066913,308
6,194953121,285
7,2154262,284
8,229095817,274
9,10981379,270
